# 🐒 Monito copión — haz hablar una foto gratis

Este notebook usa **LivePortrait** (código abierto) en la GPU gratuita de Google Colab.
Tú te grabas hablando y la foto **copia tus gestos, tu boca y tu voz**.

**Antes de empezar:** menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU → Guardar**.

Luego ejecuta las celdas **en orden**, una por una, con el botón ▶ de cada celda.

## Paso 1 — Comprobar que hay GPU
Debe aparecer una tabla que diga **Tesla T4**. Si da error, revisa que elegiste *T4 GPU* (ver arriba).

In [ ]:
!nvidia-smi

## Paso 2 — Instalar LivePortrait (tarda 5–10 minutos, solo la primera vez)
Verás mucho texto; es normal. Espera a que diga **✅ Instalación lista**.

In [ ]:
%cd /content
!git clone -q https://github.com/KwaiVGI/LivePortrait.git
%cd /content/LivePortrait
# Usamos el PyTorch que ya trae Colab; solo instalamos lo demás
!pip install -q numpy==1.26.4 pyyaml opencv-python==4.10.0.84 scipy imageio lmdb tqdm rich ffmpeg-python onnx scikit-image albumentations==1.4.10 matplotlib imageio-ffmpeg tyro==0.8.5 pykalman pillow onnxruntime-gpu transformers==4.38.0 "huggingface_hub[cli]"

# Pesos del modelo
from huggingface_hub import snapshot_download
snapshot_download("KlingTeam/LivePortrait", local_dir="pretrained_weights",
                  ignore_patterns=["*.git*", "README.md", "docs/*"])

# Componente necesario para el modo ANIMALES
%cd /content/LivePortrait/src/utils/dependencies/XPose/models/UniPose/ops
!python setup.py build install > /content/xpose_build.log 2>&1 && echo "XPose OK" || echo "⚠️ XPose falló: el modo animales no estará disponible (el modo humano sí). Detalle en /content/xpose_build.log"
%cd /content/LivePortrait
print("✅ Instalación lista")

## Paso 3 — Subir tus archivos

Ejecuta la celda y aparecerá un botón **Elegir archivos**. Sube **dos archivos a la vez**:

1. **La foto** del mono (`.jpg` o `.png`). Mientras más grande y nítida, mejor.
2. **Tu video** (`.mp4` o `.mov`) diciendo: *"Hola, ¿qué tal? Soy el monito copión"*.

**Consejos para tu video:** cara de frente, buena luz, cabeza quieta (mueve sobre todo la boca y las cejas),
5–10 segundos, empieza con la boca cerrada mirando a la cámara.

In [ ]:
import os, shutil, subprocess
from google.colab import files
os.makedirs("/content/entrada", exist_ok=True)
subidos = files.upload()
foto = video = None
for nombre in subidos:
    destino = os.path.join("/content/entrada", nombre)
    shutil.move(nombre, destino)
    ext = nombre.lower().rsplit(".", 1)[-1]
    if ext in ("jpg", "jpeg", "png", "webp"): foto = destino
    elif ext in ("mp4", "mov", "m4v", "webm", "avi"): video = destino
assert foto, "❌ Falta la foto (.jpg/.png)"
assert video, "❌ Falta el video (.mp4/.mov)"

# Pasamos el video a mp4 estándar de 25 fps (los videos de celular a veces dan problemas)
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", video, "-r", "25", "-c:v", "libx264",
                "-pix_fmt", "yuv420p", "-c:a", "aac", "/content/entrada/conduccion.mp4"], check=True)
video = "/content/entrada/conduccion.mp4"
print("Foto:", foto); print("Video:", video)

## Paso 4 — ¡Generar! (opción A: modo animales)

Es el modo pensado para animales. Está entrenado con perros y gatos, así que con un mono **puede o no funcionar**.
Si falla o el resultado se ve raro, ve al **Paso 5**.

`fuerza` controla cuánto exagera los gestos: prueba entre `1.5` y `2.0`.

In [ ]:
fuerza = 1.75
!python inference_animals.py -s "{foto}" -d "{video}" --driving_multiplier {fuerza} --no_flag_stitching --flag_crop_driving_video -o /content/resultado_animal
!ls -la /content/resultado_animal

## Paso 5 — Opción B: modo humano

Usa el modelo para caras humanas. A veces funciona con monos porque su cara se parece a la nuestra.
Si dice que **no encontró una cara**, prueba con una foto donde el mono mire más de frente.

In [ ]:
!python inference.py -s "{foto}" -d "{video}" --flag_crop_driving_video -o /content/resultado_humano
!ls -la /content/resultado_humano

## Paso 6 — Ver y descargar

Muestra los resultados y los descarga a tu computadora.
- Los archivos que terminan en `_concat.mp4` muestran **tu video + la foto + el resultado** lado a lado.
- El otro `.mp4` es **solo el resultado**, con tu voz.

In [ ]:
import glob
from IPython.display import Video, display
resultados = sorted(glob.glob("/content/resultado_*/*.mp4"))
if not resultados:
    print("❌ No hay resultados. Revisa los mensajes de error del Paso 4 o 5.")
for r in resultados:
    print("▶", r)
    display(Video(r, embed=True, width=480))
    files.download(r)

## ¿Problemas?

| Mensaje | Qué hacer |
|---|---|
| `No GPU` / `CUDA not available` | Entorno de ejecución → Cambiar tipo → **T4 GPU**, y ejecuta todo de nuevo. |
| `XPose falló` | Usa el **Paso 5** (modo humano). |
| `no face detected` / `No face` | Usa una foto donde la cara se vea de frente y más grande. |
| El resultado se ve deformado | Baja `fuerza` a `1.2`–`1.5`, o graba tu video con la cabeza más quieta. |
| Se acabó el tiempo de Colab | Es gratis con límite diario; vuelve a intentar más tarde. |